<a href="https://colab.research.google.com/github/zeneanand/IADAI201-1000442-ZENE-SOPHIE-ANAND/blob/main/FA_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

for root, dirs, files in os.walk("/content/Coffee_room_02"):
    for file in files:
        if file.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
            print(os.path.join(root, file))

In [ ]:
from IPython.display import Video

video_path = "/content/Coffee_room_02/Coffee_room_02/video (51).mp4"

Video(video_path, embed=True)

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d tuyenldvn/falldataset-imvia

Dataset URL: https://www.kaggle.com/datasets/tuyenldvn/falldataset-imvia
License(s): unknown
 87% 8.19G/9.37G [07:02<00:59, 21.3MB/s]

In [ ]:
!unzip -o falldataset-imvia.zip

In [ ]:
ls

In [ ]:
from tensorflow.keras.models import Sequential, Model

In [ ]:
import os
import cv2
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

import tensorflow as tf

from tensorflow.keras.models import Sequential, Model

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization,
    GlobalAveragePooling2D
)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Annotation file
annotation_path = "/content/Coffee_room_02/Coffee_room_02/Annotations_files/video (51).txt"

# Read annotations
with open(annotation_path, "r") as f:
    lines = f.readlines()

print("Number of annotation lines:", len(lines))
print("\nFirst 10 lines:")

for line in lines[:10]:
    print(line.strip())

In [ ]:
CLASSES = ['fall', 'walking', 'sitting', 'standing', 'normal']
NUM_CLASSES = len(CLASSES)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20


In [ ]:
data_root = '/content'
video_paths = ["/content/Coffee_room_02/Coffee_room_02/video (51).mp4"]
annotation_paths = ["/content/Coffee_room_02/Coffee_room_02/Annotations_files/video (51).txt"]

# List all top-level directories in the data_root
directories = [d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d)) and d not in ['sample_data', '.config']]

for directory in directories:
    # Construct the path to the inner directory (e.g., Coffee_room_01/Coffee_room_01/)
    inner_dir = os.path.join(data_root, directory, directory)

    if not os.path.isdir(inner_dir):
        print(f"Warning: Inner directory not found for {directory}. Skipping.")
        continue

    # Collect video files
    current_video_paths = sorted(glob.glob(os.path.join(inner_dir, '*.avi')))
    video_paths.extend(current_video_paths)

    # Collect annotation files
    annotation_folder = os.path.join(inner_dir, 'Annotation_files')
    if os.path.isdir(annotation_folder):
        current_annotation_paths = sorted(glob.glob(os.path.join(annotation_folder, '*.txt')))
        annotation_paths.extend(current_annotation_paths)
    else:
        print(f"Warning: Annotation folder not found for {directory}. Skipping annotations for this directory.")

print(f"Found {len(video_paths)} video files.")
print(f"Found {len(annotation_paths)} annotation files.")

# Create a DataFrame to store video and annotation paths
# Assuming a 1:1 mapping and consistent naming between videos and annotations
# We'll need to be careful if naming conventions differ (e.g., 'video (1).avi' vs 'video (1).txt')

# Let's try to match them based on their names (excluding extension and parent folder)
matched_data = []

video_base_names = {os.path.basename(p).split('.')[0]: p for p in video_paths}
annotation_base_names = {os.path.basename(p).split('.')[0]: p for p in annotation_paths}

for base_name, video_path in video_base_names.items():
    if base_name in annotation_base_names:
        matched_data.append({
            'video_path': video_path,
            'annotation_path': annotation_base_names[base_name]
        })
    else:
        print(f"Warning: No matching annotation found for video: {video_path}")

df_data = pd.DataFrame(matched_data)
display(df_data.head())
print(f"DataFrame created with {len(df_data)} matched video-annotation pairs.")

In [ ]:
def parse_annotation_file(annotation_path, classes_mapping):
    annotations = []
    try:
        with open(annotation_path, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 3:
                    start_frame, end_frame, event_name = parts
                    start_frame = int(start_frame)
                    end_frame = int(end_frame)
                    # Map event name to a numerical class label
                    class_id = classes_mapping.get(event_name.lower(), classes_mapping.get('normal')) # Default to 'normal' if not found
                    annotations.append({
                        'start_frame': start_frame,
                        'end_frame': end_frame,
                        'event_name': event_name.lower(),
                        'class_id': class_id
                    })
    except FileNotFoundError:
        print(f"Annotation file not found: {annotation_path}")
    except Exception as e:
        print(f"Error parsing annotation file {annotation_path}: {e}")
    return annotations

# Create a mapping from class name to integer ID
class_to_id = {cls: i for i, cls in enumerate(CLASSES)}
print("Class to ID mapping:", class_to_id)

# Example of parsing the first annotation file
if not df_data.empty:
    first_annotation_path = df_data.iloc[0]['annotation_path']
    example_annotations = parse_annotation_file(first_annotation_path, class_to_id)
    print(f"\nExample annotations from {os.path.basename(first_annotation_path)}:")
    for ann in example_annotations:
        print(ann)
else:
    print("No data to parse annotation file example.")